This notebook processes ridership data from multiple transit agencies to create representative stop-level datasets for weekdays, Saturdays, and Sundays. For each agency, the workflow generally includes:

- Adding day type based on the service date (Weekday, Saturday, Sunday), with holidays normalized or removed as needed.
- Cleaning and renaming columns to ensure consistency across datasets (stop IDs, stop names, route names, and ridership fields).
- Summing boardings and alightings across directions at each stop, and aggregating multiple time periods where applicable.
- Calculating weighted averages when ridership is reported over multiple periods, using the number of service days as weights.
- Setting a ridership basis flag to indicate that values represent reported daily boardings and/or alightings.
- Computing final average ridership per stop, route, and day type, producing clean, representative stop-level datasets ready for modeling, analysis, or comparison across agencies.

This approach standardizes ridership data across agencies with differing data formats, missing values, and service coverage, enabling consistent cross-agency analysis.

In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
bart_ridership = pd.read_excel(f"{GCS_FILE_PATH}/bart_ridership.xlsx")
big_blue_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/big_blue_bus_ridership.xlsx")
caltrain_ridership = pd.read_excel(f"{GCS_FILE_PATH}/caltrain_ridership.xlsx")
culver_citybus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/culver_citybus_ridership.xlsx")
foothill_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/foothill_transit_ridership.xlsx")
fresno_area_express_ridership = pd.read_excel(f"{GCS_FILE_PATH}/fresno_area_express_ridership.xlsx")
gold_coast_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/gold_coast_transit_ridership.xlsx")
golden_gate_park_shuttle_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_park_shuttle_ridership.xlsx")
golden_gate_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_transit_ridership.xlsx")
long_beach_transit_ridership=  pd.read_excel(f"{GCS_FILE_PATH}/long_beach_transit_ridership.xlsx")
octa_ridership = pd.read_excel(f"{GCS_FILE_PATH}/octa_ridership.xlsx")
omni_trans_ridership= pd.read_excel(f"{GCS_FILE_PATH}/omni_trans_ridership.xlsx")
riverside_transit_ridership= pd.read_excel(f"{GCS_FILE_PATH}/riverside_transit_ridership.xlsx")
sacrt_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sacrt_bus_ridership.xlsx")
samtrans_ridership = pd.read_excel(f"{GCS_FILE_PATH}/samtrans_ridership.xlsx")
santa_cruz_metro_ridership = pd.read_excel(f"{GCS_FILE_PATH}/santa_cruz_metro_ridership.xlsx")
sbmtd_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sbmtd_ridership.xlsx")
sdmts_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sdmts_ridership.xlsx")
sunline_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sunline_transit_ridership.xlsx")


In [5]:
def add_day_type(df, date_col, output_col='day_type'):
    """
    Classify each row as 'Weekday', 'Saturday', or 'Sunday' based on a specified date column.
    This is useful when daily datasets include labels such as 'Holiday' or when no day_type
    column exists. The function converts the date column, identifies the day of week, and
    standardizes the day-type classification accordingly. Either a start_date or end_date
    column can be used for datasets with daily reporting.
    """
    df = df.copy()

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    dow = df[date_col].dt.dayofweek  

    df[output_col] = np.select(
        condlist=[dow.eq(5), dow.eq(6)],
        choicelist=['Saturday', 'Sunday'],
        default='Weekday'
    )

    return df

In [6]:
# Counting number of days without holidays
def count_service_days(row):
    
    if str(row.day_type).lower() == "all":
        return (row.end_date - row.start_date).days + 1

    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum(d.weekday() < 5 for d in dates)

    elif row.day_type == "Saturday":
        return sum(d.weekday() == 5 for d in dates)

    elif row.day_type == "Sunday":
        return sum(d.weekday() == 6 for d in dates)

In [7]:
# Counting number of days with holidays
def count_service_days_wo_holidays(row):
    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum((d.weekday() < 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Saturday":
        return sum((d.weekday() == 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Sunday":
        return sum((d.weekday() == 6) and (d not in holiday_set) for d in dates)

In [8]:
def compute_averages(df, ridership_cols, group_cols):
    """
    Computes average of one or more ridership columns per group in long format.

    Parameters:
    - df: pandas DataFrame
    - ridership_cols: list of columns to average (e.g., ['daily_boardings', 'daily_alightings'])
    - group_cols: list of columns to group by (e.g., ['stop_name', 'day_type'])

    Returns:
    - DataFrame with columns: group_cols + average of each ridership column
    """
    
    if isinstance(ridership_cols, str):
        ridership_cols = [ridership_cols]  # allow single string input
    
    averaged = df.groupby(group_cols)[ridership_cols].mean().reset_index()
    
    # rename each column to indicate it's averaged
    averaged = averaged.rename(columns={col: f"average_{col}" for col in ridership_cols})
    
    return averaged

In [9]:
master_cols = [
    "organization_name",
    "route_id",
    "route_name",
    "stop_id",
    "stop_name",
    "stop_lat",
    "stop_lon",
    "day_type",
    "average_daily_boardings",
    "average_daily_alightings",
    "start_date",
    "end_date"
]

def standardize_columns(df, master_cols):
    
    # add missing columns
    for col in master_cols:
        if col not in df.columns:
            df[col] = None

    # keep only master columns and order them
    return df[master_cols]

In [43]:
bart_ridership = add_day_type(bart_ridership, date_col='start_date')

bart_ridership = bart_ridership.rename(columns={
    'Station Name': 'stop_name',
    'Stop ID': 'stop_id',
    'Date': 'date'
})

# Grouping columns
group_cols = [
    "date", "stop_id", "stop_name",
    "start_date", "end_date", "day_type"
]

# Sum boardings and alightings, with explicit output names
total_bart_ridership = (
    bart_ridership
    .groupby(group_cols, as_index=False)
    .agg(
        daily_boardings=("Entries", "sum"),
        daily_alightings=("Exits", "sum")
    )
)

# Set the ridership basis flag
total_bart_ridership["daily_ridership_basis"] = "reported_daily"


In [44]:
# Calculate average ridership per day type and stop
total_bart_ridership['daily_boardings'] = pd.to_numeric(total_bart_ridership['daily_boardings'], errors='coerce')
total_bart_ridership['daily_alightings'] = pd.to_numeric(total_bart_ridership['daily_alightings'], errors='coerce')

average_bart_ridership = compute_averages(
    df=total_bart_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id", "day_type", "stop_name"]
)

average_bart_ridership["organization_name"] = "San Francisco Bay Area Rapid Transit District"

average_bart_ridership[['start_date','end_date']] = [bart_ridership['start_date'].min(), bart_ridership['end_date'].max()]
# Remove last digit from all stop_id
average_bart_ridership['stop_id'] = average_bart_ridership['stop_id'].astype(str).str[:-1]

bart_final = standardize_columns(average_bart_ridership, master_cols)

bart_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
127,San Francisco Bay Area Rapid Transit District,None,None,90640,Millbrae,None,None,Sunday,1065.346154,1282.115385,2024-10-01,2025-09-30
1,San Francisco Bay Area Rapid Transit District,None,None,90010,12th Street / Oakland City Center,None,None,Sunday,1817.115385,1814.557692,2024-10-01,2025-09-30
110,San Francisco Bay Area Rapid Transit District,None,None,90510,Castro Valley,None,None,Weekday,1223.846743,1225.873563,2024-10-01,2025-09-30
148,San Francisco Bay Area Rapid Transit District,None,None,90950,Berryessa,None,None,Sunday,962.884615,982.692308,2024-10-01,2025-09-30
87,San Francisco Bay Area Rapid Transit District,None,None,90380,Pittsburg / Bay Point,None,None,Saturday,835.288462,835.576923,2024-10-01,2025-09-30


In [12]:
big_blue_bus_ridership.columns = big_blue_bus_ridership.columns.str.lower()
big_blue_bus_ridership['service_day'] = big_blue_bus_ridership['service_day'].str.strip().str.capitalize()


big_blue_bus_ridership = big_blue_bus_ridership.rename(columns={
    'route_number': 'route_id',
    'service_day': 'day_type',
    'average_daily_boardings':'average_daily_boardings_period',
    'average_daily_alightings':'average_daily_alightings_period'    
})


# Sum boardings/alightings across directions within same stop & period
total_big_blue_bus_ridership = (
    big_blue_bus_ridership
    .groupby(['day_type','route_id','route_name',
              'stop_id','stop_name','stop_lat','stop_lon',
              'start_date','end_date'])
    .agg({
        'average_daily_boardings_period':'sum',
        'average_daily_alightings_period':'sum'
    })
    .reset_index()
)


In [13]:
big_blue_bus_ridership['start_date'] = pd.to_datetime(big_blue_bus_ridership['start_date'])
big_blue_bus_ridership['end_date'] = pd.to_datetime(big_blue_bus_ridership['end_date'])

total_big_blue_bus_ridership["n_days"] = total_big_blue_bus_ridership.apply(count_service_days, axis=1)

# Taking weighted average of the ridership across 4 different time periods
averaged_total_big_blue_bus_ridership = (
    total_big_blue_bus_ridership
    .groupby(['day_type','route_id','route_name', 'stop_name',
              'stop_id','stop_lat','stop_lon'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['average_daily_alightings_period'], weights=x['n_days'])
    }))
    .reset_index()
)


# Set the ridership basis flag
averaged_total_big_blue_bus_ridership["daily_ridership_basis"] = "reported_avg_daily"
averaged_total_big_blue_bus_ridership["organization_name"] = "Big Blue Bus"

averaged_total_big_blue_bus_ridership[['start_date','end_date']] = [big_blue_bus_ridership['start_date'].min(), big_blue_bus_ridership['end_date'].max()]

big_blue_bus_ridership_final = standardize_columns(averaged_total_big_blue_bus_ridership, master_cols)
big_blue_bus_ridership_final.sample(5)

/tmp/ipykernel_6023/404159155.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2876,Big Blue Bus,41,17th St Station - SMC,1022,20TH SB/IDAHO FS,34.035309,-118.487880,Weekday,3.219868,2.577853,2024-08-01,2025-11-30
2800,Big Blue Bus,23,Lincoln Blvd/LAX Rapid,2692,LINCOLN NB/VENICE NS,33.993979,-118.452311,Weekday,7.807586,6.844354,2024-08-01,2025-11-30
486,Big Blue Bus,9,Pacific Palisades,2762,SUNSET WB/ARNO NS,34.043733,-118.544475,Saturday,0.026125,1.129815,2024-08-01,2025-11-30
1084,Big Blue Bus,7,Pico Blvd,2792,CRENSHAW SB/OLYMPIC FS,34.054287,-118.323305,Sunday,42.303109,3.117087,2024-08-01,2025-11-30
287,Big Blue Bus,7,Pico Blvd,2327,PICO EB/VETERAN NS,34.038602,-118.431467,Saturday,15.626020,8.722443,2024-08-01,2025-11-30


In [14]:
caltrain_ridership = caltrain_ridership.rename(columns={
    'Origin Station': 'stop_name',
    'Date Type': 'day_type',
    'Average Ridership':'average_daily_boardings_period',  
})

caltrain_ridership = caltrain_ridership[
    caltrain_ridership["day_type"] != "Holiday"
]

# Getting number of days and using US Federal Holidays
start_date_caltrain = '2023-11-01'
end_date_caltrain = '2025-07-31'

#Getting all the federal holidays between the start and end date
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=start_date_caltrain, end=end_date_caltrain)

# Setting holidays to calculate the number of weekdays, saturday and sunday without holidays 
holiday_set = set(holidays)

caltrain_ridership["start_date"] = pd.to_datetime(caltrain_ridership["start_date"])
caltrain_ridership["end_date"] = pd.to_datetime(caltrain_ridership["end_date"])

caltrain_ridership["n_days"] = caltrain_ridership.apply(count_service_days_wo_holidays, axis=1)

In [15]:
# Taking weighted average of the ridership across 20 months time periods
averaged_caltrain_ridership = (
    caltrain_ridership
    .groupby(['day_type','stop_name'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days'])
    }))
    .reset_index()
)

# Set the ridership basis flag
averaged_caltrain_ridership["daily_ridership_basis"] = "reported_avg_daily"
averaged_caltrain_ridership["organization_name"] = "Caltrain"

averaged_caltrain_ridership[['start_date','end_date']] = [caltrain_ridership['start_date'].min(), caltrain_ridership['end_date'].max()]

caltrain_ridership_final = standardize_columns(averaged_caltrain_ridership, master_cols)

caltrain_ridership_final.sample(5)

/tmp/ipykernel_6023/3722569284.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
18,Caltrain,None,None,None,Redwood City,None,None,Saturday,799.444990,None,2023-11-01,2025-07-31
86,Caltrain,None,None,None,Santa Clara,None,None,Weekday,718.467622,None,2023-11-01,2025-07-31
3,Caltrain,None,None,None,Blossom Hill,None,None,Saturday,0.199045,None,2023-11-01,2025-07-31
28,Caltrain,None,None,None,Sunnyvale,None,None,Saturday,753.458120,None,2023-11-01,2025-07-31
64,Caltrain,None,None,None,Broadway,None,None,Weekday,0.000000,None,2023-11-01,2025-07-31


In [16]:
culver_citybus_ridership["daily_ridership_basis"] = "reported_avg_daily"
culver_citybus_ridership["Route"] = culver_citybus_ridership["Route"].str.split("-", n=1).str[1]


culver_citybus_ridership = culver_citybus_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'route_id': 'route_id',
    'Route': 'route_name',
})

group_cols = [
    "route_name", "stop_id", "stop_name", "route_id", "day_type", "daily_ridership_basis"]

# Sum AVG On and AVG Off
sum_culver_citybus_ridership = (culver_citybus_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("AVG On", "sum"),
        average_daily_alightings=("AVG Off", "sum")
    )
)

sum_culver_citybus_ridership["organization_name"] = "Culver City Bus"

sum_culver_citybus_ridership[['start_date','end_date']] = [culver_citybus_ridership['start_date'].min(), culver_citybus_ridership['end_date'].max()]

culver_citybus_final = standardize_columns(sum_culver_citybus_ridership, master_cols)

culver_citybus_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
759,Culver City Bus,Rapid 6,Sepulveda Boulevard,688,SepulvedaBlvd/SlausonAve,None,None,Weekday,24.5,108.1,2025-07-14,2025-08-25
492,Culver City Bus,4,Jefferson Boulevard,437,Jefferson Blvd/Indian Wood Rd,None,None,Weekday,1.4,5.3,2025-07-14,2025-08-25
66,Culver City Bus,3,Crosstown,329,Motor Ave/Woodbine St,None,None,Sunday,0.2,2.5,2025-07-14,2025-08-25
976,Culver City Bus,1,Washington Boulevard,163,WashingtonBlvd/InglewoodBlvd,None,None,Weekday,43.1,65.9,2025-07-14,2025-08-25
373,Culver City Bus,1c1,Culver City Downtown Circulator,148,Washington Blvd/Robertson Blvd,None,None,Saturday,0.0,0.5,2025-07-14,2025-08-25


In [17]:
# Add day_type from the date column
foothill_transit_ridership = add_day_type(foothill_transit_ridership, date_col="date")

foothill_transit_ridership = foothill_transit_ridership.rename(columns={
    'route_short_name': 'route_id',
    'stop_code': 'stop_id',
})

# Grouping columns
group_cols = [
    "date", "route_id", "stop_id", "stop_lat", "stop_lon",
    "start_date", "end_date", "day_type"
]

# Sum boardings and alightings, with explicit output names
total_ridership_foothill = (
    foothill_transit_ridership
    .groupby(group_cols, as_index=False)
    .agg(
        daily_boardings=("boardings", "sum"),
        daily_alightings=("alightings", "sum")
    )
)

# Set the ridership basis flag
total_ridership_foothill["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_ridership_foothill['daily_boardings'] = pd.to_numeric(total_ridership_foothill['daily_boardings'], errors='coerce')
total_ridership_foothill['daily_alightings'] = pd.to_numeric(total_ridership_foothill['daily_alightings'], errors='coerce')

average_foothill_ridership = compute_averages(
    df=total_ridership_foothill,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["route_id", "stop_id", 
                "stop_lat", "stop_lon", "day_type"]
)

average_foothill_ridership["organization_name"] = "Foothill Transit"

average_foothill_ridership[['start_date','end_date']] = [foothill_transit_ridership['start_date'].min(), foothill_transit_ridership['end_date'].max()]
foothill_final = standardize_columns(average_foothill_ridership, master_cols)

foothill_final.sample(5)


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
3786,Foothill Transit,284,None,1516,None,34.135767,-117.865379,Saturday,3.577778,0.644444,2024-07-01,2025-06-30
2838,Foothill Transit,274,None,2751,None,34.028898,-118.026235,Weekday,2.082873,0.464088,2024-07-01,2025-06-30
1973,Foothill Transit,195,None,2145,None,34.033441,-117.749804,Sunday,2.425532,1.617021,2024-07-01,2025-06-30
4309,Foothill Transit,289,None,1627,None,34.030467,-117.837428,Saturday,1.454545,0.181818,2024-07-01,2025-06-30
1275,Foothill Transit,188,None,2264,None,34.128882,-117.881123,Weekday,10.965517,3.605364,2024-07-01,2025-06-30


In [18]:
# Add day_type from the date column
fresno_area_express_ridership = add_day_type(fresno_area_express_ridership, date_col="Date")

fresno_area_express_ridership["daily_ridership_basis"] = "reported_daily"


fresno_area_express_ridership = fresno_area_express_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopLabel': 'stop_name',
    'ProjectedBoarding': 'daily_boardings',
    'ProjectedAlighting': 'daily_alightings'
})

# Calculate average ridership per day type and stop
fresno_area_express_ridership['daily_boardings'] = pd.to_numeric(fresno_area_express_ridership['daily_boardings'], errors='coerce')
fresno_area_express_ridership['daily_alightings'] = pd.to_numeric(fresno_area_express_ridership['daily_alightings'], errors='coerce')

average_fresno_area_express_ridership = compute_averages(
    df=fresno_area_express_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id",  "stop_name", "day_type"]
)

average_fresno_area_express_ridership["organization_name"] = "Fresno County"

average_fresno_area_express_ridership[['start_date','end_date']] = [fresno_area_express_ridership['start_date'].min(), fresno_area_express_ridership['end_date'].max()]

fresno_final = standardize_columns(average_fresno_area_express_ridership, master_cols)

fresno_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
3641,Fresno County,None,None,1682,4175 E SHIELDS,None,None,Weekday,50.070782,11.740158,2024-09-01,2025-08-31
1896,Fresno County,None,None,897,NW WISHON - WELDON,None,None,Weekday,28.770820,41.795143,2024-09-01,2025-08-31
3645,Fresno County,None,None,1684,NW SHIELDS - MILLBROOK,None,None,Saturday,7.961234,3.392426,2024-09-01,2025-08-31
3357,Fresno County,None,None,1573,SE JENSEN - MARTIN LUTHER KING,None,None,Saturday,9.339603,5.672676,2024-09-01,2025-08-31
3621,Fresno County,None,None,1675,SE CHESTNUT - PRINCETON,None,None,Saturday,1.096014,1.386984,2024-09-01,2025-08-31


In [19]:
gold_coast_transit_ridership = gold_coast_transit_ridership.rename(columns={
    'day_of_week': 'day_type',
    'route': 'route_id',
    'lat': 'stop_lat',
    'lon': 'stop_lon'
})

group_cols = ["day_type", "route_id", "stop_id", "stop_name",  "stop_lat", "stop_lon", "start_date", "end_date"]

# Sum AVG On 
total_gold_coast_transit_ridership = (gold_coast_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("total_on", "sum"),
        daily_alightings=("total_off", "sum"),
    )
)

total_gold_coast_transit_ridership['start_date'] = pd.to_datetime(total_gold_coast_transit_ridership['start_date'])
total_gold_coast_transit_ridership['end_date'] = pd.to_datetime(total_gold_coast_transit_ridership['end_date'])

total_gold_coast_transit_ridership["n_days"] = total_gold_coast_transit_ridership.apply(count_service_days, axis=1)

In [20]:
# Taking weighted average of the ridership across 4 different time periods
averaged_gold_coast_transit_ridership = (
    total_gold_coast_transit_ridership
    .groupby(['day_type','route_id', 'stop_name',
              'stop_id','stop_lat','stop_lon'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['daily_boardings'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['daily_alightings'], weights=x['n_days'])
    }))
    .reset_index()
)

averaged_gold_coast_transit_ridership["organization_name"] = "Gold Coast Transit"

averaged_gold_coast_transit_ridership[['start_date','end_date']] = [gold_coast_transit_ridership['start_date'].min(), gold_coast_transit_ridership['end_date'].max()]
golden_coast_transit_final = standardize_columns(averaged_gold_coast_transit_ridership, master_cols)

golden_coast_transit_final.sample(5)

/tmp/ipykernel_6023/93958713.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
563,Gold Coast Transit,17,None,ROSSLA1,Rose & Santa Lucia,34.201636,-119.159878,Weekday,19.0,22.0,2025-05-01,2025-05-31
145,Gold Coast Transit,6,None,MAISEA1,Main & Seaward,34.276358,-119.265088,Weekday,27.0,34.0,2025-05-01,2025-05-31
376,Gold Coast Transit,11,None,TPHSCA1,Telephone & Scandia,34.278164,-119.160677,Weekday,8.0,3.0,2025-05-01,2025-05-31
111,Gold Coast Transit,6,None,CSTGNZ2,C & Gonzales,34.220640,-119.180900,Weekday,22.0,36.0,2025-05-01,2025-05-31
375,Gold Coast Transit,11,None,TPHSAT2,Telephone & Saticoy,34.279470,-119.158566,Weekday,23.0,14.0,2025-05-01,2025-05-31


In [21]:
# Add day_type from the date column
golden_gate_park_shuttle_ridership = add_day_type(golden_gate_park_shuttle_ridership, date_col="Date")

group_cols = [
    "Date", "day_type", "Stop", "Stop ID", "start_date", "end_date"]

# Sum AVG On 
total_golden_gate_park_shuttle_ridership = (golden_gate_park_shuttle_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Ridership", "sum"),
    )
)

total_golden_gate_park_shuttle_ridership["daily_ridership_basis"] = "reported_daily"

total_golden_gate_park_shuttle_ridership = total_golden_gate_park_shuttle_ridership.rename(columns={
    'Stop': 'stop_name',
    'Date': 'date',
    'Stop ID': 'stop_id'
})

# Calculate average ridership per day type and stop
total_golden_gate_park_shuttle_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_park_shuttle_ridership['daily_boardings'], errors='coerce')
average_golden_gate_park_shuttle_ridership = compute_averages(
    df=total_golden_gate_park_shuttle_ridership,
    ridership_cols=["daily_boardings"],
    group_cols=["stop_name", "day_type", "stop_id"]
)


average_golden_gate_park_shuttle_ridership["organization_name"] = "Golden Gate Park Shuttle"

average_golden_gate_park_shuttle_ridership[['start_date','end_date']] = [golden_gate_park_shuttle_ridership['start_date'].min(), 
                                                                         golden_gate_park_shuttle_ridership['end_date'].max()]
golden_gate_parkshuttle_final = standardize_columns(average_golden_gate_park_shuttle_ridership, master_cols)

golden_gate_parkshuttle_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
43,Golden Gate Park Shuttle,None,None,7603,Rose Garden WB,None,None,Sunday,9.634615,None,2024-07-01,2025-06-30
21,Golden Gate Park Shuttle,None,None,7613,Conservatory of Flowers WB,None,None,Saturday,20.730769,None,2024-07-01,2025-06-30
2,Golden Gate Park Shuttle,None,None,7607,10th Ave/ De Young EB,None,None,Weekday,5.862069,None,2024-07-01,2025-06-30
44,Golden Gate Park Shuttle,None,None,7603,Rose Garden WB,None,None,Weekday,9.157088,None,2024-07-01,2025-06-30
45,Golden Gate Park Shuttle,None,None,7615,Tennis Center/ Dalia Dell EB,None,None,Saturday,11.192308,None,2024-07-01,2025-06-30


In [22]:
# Add day_type from the date column
golden_gate_transit_ridership = add_day_type(golden_gate_transit_ridership, date_col="date")

golden_gate_transit_ridership = golden_gate_transit_ridership.rename(columns={
    'STOP_NUMBER': 'stop_id',
    'STOP_NAME': 'stop_name',
    'ROUTE': 'route_id',
})

# Remove everything in parentheses (including the parentheses) at the end of the string
golden_gate_transit_ridership['stop_name'] = golden_gate_transit_ridership['stop_name'].str.replace(r'\s*\(.*\)$', '', regex=True)

group_cols = [
    "day_type", "route_id", "stop_id", "stop_name", "end_date", "start_date"]

# Sum AVG On and AVG Off
total_golden_gate_transit_ridership = (golden_gate_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("BOARDINGS", "sum"),
        daily_alightings=("ALIGHTINGS", "sum")
    )
)

total_golden_gate_transit_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_golden_gate_transit_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_transit_ridership['daily_boardings'], errors='coerce')
total_golden_gate_transit_ridership['daily_alightings'] = pd.to_numeric(total_golden_gate_transit_ridership['daily_alightings'], errors='coerce')

average_golden_gate_transit_ridership = compute_averages(
    df=total_golden_gate_transit_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["route_id", "stop_id", "stop_name", "day_type"]
)

average_golden_gate_transit_ridership["organization_name"] = "Golden Gate Transit"
average_golden_gate_transit_ridership[['start_date','end_date']] = [golden_gate_transit_ridership['start_date'].min(), 
                                                                         golden_gate_transit_ridership['end_date'].max()]

golden_gate_transit_final = standardize_columns(average_golden_gate_transit_ridership, master_cols)

golden_gate_transit_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
819,Golden Gate Transit,150,None,40028,Van Ness Ave & Bush St,None,None,Sunday,13.500000,4.000000,2025-09-01,2025-09-30
1312,Golden Gate Transit,172X,None,80027,VTP 101 SB @ Hwy 37,None,None,Weekday,0.000000,0.000000,2025-09-01,2025-09-30
161,Golden Gate Transit,101,None,41206,E Washington St & Lakeville St,None,None,Sunday,2.250000,3.000000,2025-09-01,2025-09-30
1346,Golden Gate Transit,580,None,41037,Bellam Blvd & I-580,None,None,Weekday,23.000000,3.545455,2025-09-01,2025-09-30
734,Golden Gate Transit,132,None,40039,Van Ness Ave & Chestnut St,None,None,Weekday,20.285714,7.285714,2025-09-01,2025-09-30


In [23]:
long_beach_transit_ridership["daily_ridership_basis"] = "reported_avg_daily"

group_cols = [
    "DayType", "Route", "StopID", "StopName", "end_date", "start_date", "daily_ridership_basis"]

# Sum AVG On and AVG Off
total_long_beach_transit_ridership = (long_beach_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("Boardings", "sum"),
        average_daily_alightings=("Alightings", "sum")
    )
)

total_long_beach_transit_ridership = total_long_beach_transit_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopName': 'stop_name',
    'DayType': 'day_type',
    'Route': 'route_id',
})

total_long_beach_transit_ridership["organization_name"] = "Long Beach Transit"

total_long_beach_transit_ridership[['start_date','end_date']] = [long_beach_transit_ridership['start_date'].min(), 
                                                                         long_beach_transit_ridership['end_date'].max()]

long_beach_transit_final = standardize_columns(total_long_beach_transit_ridership, master_cols)

long_beach_transit_final.sample(5)


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
3819,Long Beach Transit,91,None,1019,7th & Ximeno SE,None,None,Sunday,3.325056,11.608070,2024-07-01,2025-06-30
7709,Long Beach Transit,112,None,1423,Alamitos & Broadway SW,None,None,Weekday,1.514667,7.566489,2024-07-01,2025-06-30
8899,Long Beach Transit,192,None,1618,South & Obispo SW,None,None,Weekday,2.514887,3.770430,2024-07-01,2025-06-30
8696,Long Beach Transit,191,None,129,Wardlow & Webster SE,None,None,Weekday,24.723297,7.566271,2024-07-01,2025-06-30
1019,Long Beach Transit,71,None,899,Orange & Burnett NW,None,None,Saturday,1.596396,4.785714,2024-07-01,2025-06-30


In [24]:
octa_ridership = octa_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'Route': 'route_name'

})

octa_ridership["stop_name"] = octa_ridership["stop_name"].str.split("-", n=1).str[1]
octa_ridership["route_name"] = octa_ridership["route_name"].str.split("-", n=1).str[1]

group_cols = [
    "day_type", "route_name", "stop_id", "route_id", "stop_name", "end_date", "start_date"]

# Sum AVG On and AVG Off
octa_ridership_sum = (octa_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("APC Boarding", "sum"),
        average_daily_alightings=("APC Alighting", "sum")
    )
)

octa_ridership_sum["organization_name"] = "Orange County Transportation Authority"

octa_ridership_sum[['start_date','end_date']] = [octa_ridership['start_date'].min(), 
                                                                         octa_ridership['end_date'].max()]

octa_final = standardize_columns(octa_ridership_sum, master_cols)

octa_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
1311,Orange County Transportation Authority,35,Fullerton - Huntington Beach,131,BROOKHURST-CRESCENT,None,None,weekday,28,72,2025-02-04,2025-02-04
1650,Orange County Transportation Authority,35,Fullerton - Huntington Beach,7382,BROOKHURST-BOLSA,None,None,weekday,60,103,2025-02-04,2025-02-04
1465,Orange County Transportation Authority,35,Fullerton - Huntington Beach,2184,BROOKHURST-JENNRICH,None,None,weekday,16,10,2025-02-04,2025-02-04
4407,Orange County Transportation Authority,42,Orange - Seal Beach,3107,NORWALK-CIVIC CENTER,None,None,weekday,24,153,2025-02-04,2025-02-04
3867,Orange County Transportation Authority,60,Long Beach - Tustin,2542,WESTMINSTER-TAFT,None,None,weekday,12,14,2025-02-04,2025-02-04


In [25]:
omni_trans_ridership= omni_trans_ridership.rename(columns={
    'Route': 'route_id',
    'Stop Name': 'stop_name',
    'avg_boardings': 'average_daily_boardings',
    'avg_alightings': 'average_daily_alightings',
    'Stop Id': 'stop_id'
})

omni_trans_ridership["organization_name"] = "Omnitrans"

omni_trans_ridership_filtered = omni_trans_ridership[['route_id', 'stop_name', 'stop_id',  'average_daily_boardings', 'average_daily_alightings', 'organization_name', 'day_type']]

omni_trans_ridership_filtered['stop_id'] = omni_trans_ridership_filtered['stop_id'].apply(
    lambda x: None if pd.isna(x) else int(x)
)

omni_trans_ridership_filtered[['start_date','end_date']] = [omni_trans_ridership['start_date'].min(), 
                                                                         omni_trans_ridership['end_date'].max()]
omni_final = standardize_columns(omni_trans_ridership_filtered, master_cols)

omni_final.sample(5)

/tmp/ipykernel_6023/1053658925.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  omni_trans_ridership_filtered['stop_id'] = omni_trans_ridership_filtered['stop_id'].apply(
/tmp/ipykernel_6023/1053658925.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  omni_trans_ridership_filtered[['start_date','end_date']] = [omni_trans_ridership['start_date'].min(),
/tmp/ipykernel_6023/1053658925.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_in

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
1866,Omnitrans,3,None,NaN,MEDICAL CENTER @ 11 TH,None,None,all,12.843836,8.717808,2023-07-01,2026-06-30
3302,Omnitrans,1,None,5251.0,Mt Vernon @ La Cadena,None,None,all,2.334247,1.934247,2023-07-01,2026-06-30
2011,Omnitrans,6,None,5294.0,Valencia @ Highland,None,None,all,2.791781,3.463014,2023-07-01,2026-06-30
588,Omnitrans,15,None,6536.0,9th @ Sterling,None,None,all,3.841530,3.890710,2023-07-01,2026-06-30
2678,Omnitrans,66,None,NaN,WEST VALLEY,None,None,all,8.621918,7.860274,2023-07-01,2026-06-30


In [26]:
riverside_transit_ridership = add_day_type(riverside_transit_ridership, date_col='date')

group_cols = [
    "date", "Stop ID", "Route", "start_date", 
    "end_date", "day_type"]

# Sum AVG On and AVG Off
total_riverside_transit_ridership = (riverside_transit_ridership.groupby(
    group_cols, as_index=False).agg(
    daily_boardings = ('ridership', 'sum'))
)

total_riverside_transit_ridership = total_riverside_transit_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Route': 'route_id'

})

In [27]:
total_riverside_transit_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_riverside_transit_ridership['daily_boardings'] = pd.to_numeric(total_riverside_transit_ridership['daily_boardings'], errors='coerce')
average_riverside_transit_ridership = compute_averages(
    df=total_riverside_transit_ridership,
    ridership_cols=["daily_boardings"],
    group_cols=["route_id", "stop_id", "day_type"]
)

average_riverside_transit_ridership["organization_name"] = "Riverside Transit"

average_riverside_transit_ridership[['start_date','end_date']] = [riverside_transit_ridership['start_date'].min(), 
                                                                         riverside_transit_ridership['end_date'].max()]

riverside_final = standardize_columns(average_riverside_transit_ridership, master_cols)

riverside_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
5342,Riverside Transit,23,None,2529,None,None,None,Weekday,1.950820,None,2025-01-01,2025-10-31
7899,Riverside Transit,79,None,3393,None,None,None,Weekday,9.279570,None,2025-01-01,2025-10-31
4067,Riverside Transit,19,None,2090,None,None,None,Saturday,11.000000,None,2025-01-01,2025-10-31
3493,Riverside Transit,15,None,1848,None,None,None,Weekday,4.400000,None,2025-01-01,2025-10-31
7657,Riverside Transit,74,None,2741,None,None,None,Weekday,1.968992,None,2025-01-01,2025-10-31


In [28]:
sacrt_bus_ridership = sacrt_bus_ridership.rename(columns={
    'route_long_name': 'route_name',
})

group_cols = [
    "stop_id", "stop_lat", "stop_lon", "stop_name", "route_name", "route_id", "day_type"]

# Combining across directions
sacrt_bus_ridership_sum = (sacrt_bus_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("average_boardings", "sum"),
        average_daily_alightings=("average_alightings", "sum")
    )
)

# Replace day_type
sacrt_bus_ridership_sum["day_type"] = sacrt_bus_ridership_sum["day_type"].replace("daily", "all")

sacrt_bus_ridership_sum["organization_name"] = "SacRT Bus"

sacrt_bus_ridership_sum[['start_date','end_date']] = [sacrt_bus_ridership['start_date'].min(), 
                                                                         sacrt_bus_ridership['end_date'].max()]

sacrt_bus_final = standardize_columns(sacrt_bus_ridership_sum, master_cols)

sacrt_bus_final.sample(5)


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2187,SacRT Bus,062,FREEPORT,2239,FREEPORT BLVD & 6TH AVE (NB),38.546756,-121.490027,all,1,1,2023-09-01,2023-12-31
3482,SacRT Bus,102,RIVERSIDE COMMUTER,4262,RIVERSIDE BLVD & PARK RIVIERA WAY (EB),38.503182,-121.547050,weekday,0,0,2023-09-01,2023-12-31
3644,SacRT Bus,105,ELSIE,5109,FRANKLIN BLVD & ARMADALE WY (SB),38.470642,-121.449028,weekday,0,0,2023-09-01,2023-12-31
1125,SacRT Bus,206,12TH AVE - SUTTERVILLE RD,1050,LAND PARK DR & 7TH AVE (NB),38.548384,-121.499283,weekday,0,0,2023-09-01,2023-12-31
2054,SacRT Bus,061,FRUITRIDGE,2006,75TH ST & ELDERCREEK RD (SB),38.510277,-121.415985,all,5,6,2023-09-01,2023-12-31


In [29]:
samtrans_ridership = add_day_type(samtrans_ridership, date_col='APC Date')

samtrans_ridership = samtrans_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'Route': 'route_id',
    'Lat': 'stop_lat',
    'Lon': 'stop_lon',
    'APC Date': 'date',
    'Ons': 'daily_boardings',
    'Offs': 'daily_alightings',
})

samtrans_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
samtrans_ridership['daily_boardings'] = pd.to_numeric(samtrans_ridership['daily_boardings'], errors='coerce')
samtrans_ridership['daily_alightings'] = pd.to_numeric(samtrans_ridership['daily_alightings'], errors='coerce')

average_samtrans_ridership = compute_averages(
    df=samtrans_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["route_id", "stop_id", "stop_name", "stop_lat", "stop_lon", "day_type"]
)

average_samtrans_ridership["organization_name"] = "Samtrans"

average_samtrans_ridership[['start_date','end_date']] = [samtrans_ridership['start_date'].min(), 
                                                                         samtrans_ridership['end_date'].max()]

samtrans_final = standardize_columns(average_samtrans_ridership, master_cols)

samtrans_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
1970,Samtrans,142,None,335098,Huntington Ave & Euclid Ave,37.632301,-122.412628,Weekday,1.809524,3.047619,2025-08-01,2025-08-31
128,Samtrans,110,None,311119,Oceana Blvd & Brighton Rd,37.630767,-122.489014,Saturday,6.800000,7.000000,2025-08-01,2025-08-31
5034,Samtrans,46,None,340110,Trousdale Dr & Magnolia Ave,37.593761,-122.383972,Weekday,0.500000,1.666667,2025-08-01,2025-08-31
5925,Samtrans,ECR,None,332216,Mission St & Alp Ave,37.703449,-122.462684,Weekday,2.904762,35.476190,2025-08-01,2025-08-31
1497,Samtrans,130,None,332097,E Market St & Hillside Blvd,37.690963,-122.459854,Sunday,11.400000,2.800000,2025-08-01,2025-08-31


In [30]:
group_cols = [
    "Route", "Route Name", "Stop ID",  "Stop Name", "day_type", "start_date", 
    "end_date"]

# Sum AVG On and AVG Off
total_sdmts_ridership = (sdmts_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("Average On", "sum"),
        average_daily_alightings=("Average Off", "sum")
    )
)

total_sdmts_ridership["Route Name"] = total_sdmts_ridership["Route Name"].str.split(":", n=1).str[1]

total_sdmts_ridership = total_sdmts_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Route': 'route_id',
    'Route Name': 'route_name',
    'Stop Name': 'stop_name',
})

total_sdmts_ridership["daily_ridership_basis"] = "reported_avg_daily"

total_sdmts_ridership["organization_name"] = "SDMTS"

total_sdmts_ridership[['start_date','end_date']] = [sdmts_ridership['start_date'].min(), 
                                                                         sdmts_ridership['end_date'].max()]

sdmts_final = standardize_columns(total_sdmts_ridership, master_cols)

sdmts_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
7418,SDMTS,709,H St TC-Otay Ranch Town Center,30194,East H St & Terra Nova Dr,None,None,Saturday,4.049738,2.148409,2024-09-01,2025-01-25
9318,SDMTS,874,El Cajon Eastside Shuttle Clockwise,40445,Washington Av & Lincoln Av,None,None,Sunday,5.815789,1.563467,2024-09-01,2025-01-25
4086,SDMTS,31,UTC-Mira Mesa,12841,Black Mountain Rd & Activity Rd,None,None,Weekday,0.831466,1.004859,2024-09-01,2025-01-25
9704,SDMTS,901,Iris Transit Center-Downtown,60182,Imperial Beach Bl & Granger St,None,None,Sunday,2.641390,2.545540,2024-09-01,2025-01-25
11629,SDMTS,932,Iris Transit Center-8th St. Transit Center,60409,Broadway & D St,None,None,Saturday,2.908007,13.387881,2024-09-01,2025-01-25


In [31]:
santa_cruz_metro_ridership = santa_cruz_metro_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'Boardings': 'average_daily_boardings',
    'Alightings': 'average_daily_alightings'    
})

santa_cruz_metro_ridership["organization_name"] = "Santa Cruz Metro"

santa_cruz_metro_ridership[['start_date','end_date']] = [santa_cruz_metro_ridership['start_date'].min(), 
                                                                         santa_cruz_metro_ridership['end_date'].max()]

santa_cruz_final = standardize_columns(santa_cruz_metro_ridership, master_cols)

santa_cruz_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
675,Santa Cruz Metro,None,None,1840,Soquel Dr + Cabrillo College Dr,None,None,all,114.208219,93.378082,2024-07-01,2025-06-30
214,Santa Cruz Metro,None,None,1363,E Cliff Dr (East Cliff Village),None,None,all,7.564384,3.550685,2024-07-01,2025-06-30
433,Santa Cruz Metro,None,None,1564,Hwy 9 + Monaco Ln,None,None,all,0.167123,1.408219,2024-07-01,2025-06-30
407,Santa Cruz Metro,None,None,1537,Hwy 9 (Highlands Park),None,None,all,0.389041,0.838356,2024-07-01,2025-06-30
595,Santa Cruz Metro,None,None,1094,S Green Valley Rd + Hope Dr (Kingdom Hall),None,None,all,0.534247,3.452055,2024-07-01,2025-06-30


In [32]:
sbmtd_ridership = sbmtd_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'avg_boardings': 'daily_boardings',
    'avg_ridership': 'daily_alightings'    
})


sbmtd_ridership["n_days"] = sbmtd_ridership.apply(count_service_days, axis=1)

# Taking weighted average of the ridership across 12 months
averaged_sbmtd_ridership = (
    sbmtd_ridership
    .groupby(['stop_id', 'stop_name', 'day_type'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['Ridership by Stop_Boardings'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['Ridership by Stop_Alightings'], weights=x['n_days'])
    }))
    .reset_index()
)



averaged_sbmtd_ridership["organization_name"] = "SBMTD"

averaged_sbmtd_ridership[['start_date','end_date']] = [sbmtd_ridership['start_date'].min(), 
                                                                         sbmtd_ridership['end_date'].max()]

sbmtd_final = standardize_columns(averaged_sbmtd_ridership, master_cols)

sbmtd_final.sample(5)

/tmp/ipykernel_6023/1734702637.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
147,SBMTD,None,None,185,Calle Real & Turnpike West,None,None,all,276.490411,275.515068,2024-11-01,2025-10-31
265,SBMTD,None,None,312,Milpas & Gutierrez,None,None,all,284.257534,289.169863,2024-11-01,2025-10-31
450,SBMTD,None,None,539,Meigs & La Coronilla,None,None,all,31.838356,43.238356,2024-11-01,2025-10-31
181,SBMTD,None,None,223,Cliff & Alan,None,None,all,30.090411,15.526027,2024-11-01,2025-10-31
142,SBMTD,None,None,180,Anacapa & Sola,None,None,all,186.660274,905.038356,2024-11-01,2025-10-31


In [45]:
all_ridership = pd.concat([
    bart_final,
    big_blue_bus_ridership_final,
    caltrain_ridership_final,
    culver_citybus_final,
    foothill_final,
    fresno_final,
    golden_coast_transit_final,
    golden_gate_parkshuttle_final,
    golden_gate_transit_final,
    long_beach_transit_final,
    octa_final,
    omni_final,
    riverside_final,
    sacrt_bus_final,
    samtrans_final,
    sdmts_final,
    santa_cruz_final,
    sbmtd_final,
], ignore_index=True)

/tmp/ipykernel_6023/1180593662.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_ridership = pd.concat([


In [46]:
all_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71703 entries, 0 to 71702
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   organization_name         71703 non-null  object        
 1   route_id                  65173 non-null  object        
 2   route_name                27000 non-null  object        
 3   stop_id                   71293 non-null  object        
 4   stop_name                 56894 non-null  object        
 5   stop_lat                  21395 non-null  float64       
 6   stop_lon                  21395 non-null  float64       
 7   day_type                  71703 non-null  object        
 8   average_daily_boardings   71703 non-null  float64       
 9   average_daily_alightings  63461 non-null  float64       
 10  start_date                71703 non-null  datetime64[ns]
 11  end_date                  71703 non-null  datetime64[ns]
dtypes: datetime64[ns](

In [82]:
all_ridership.to_csv(f"{GCS_FILE_PATH}/AHSC_2026/ridership_all.csv", index=False)